In [2]:
# ---------------------------------------------------------------
# FOG PREDICTION USING MACHINE LEARNING
# Includes: Data Loading, Cleaning, Transformation, Training
# ---------------------------------------------------------------

# Import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# ---------------------------------------------------------------
# 1. LOAD YOUR DATA
# Replace 'weather.csv' with your file name
# The dataset must contain: Temperature, DewPoint, Visibility
# ---------------------------------------------------------------

df = pd.read_csv("weather.csv")

# ---------------------------------------------------------------
# 2. DISPLAY FIRST FEW ROWS
# ---------------------------------------------------------------
print("Sample Data:")
print(df.head())

# ---------------------------------------------------------------
# 3. DATA CLEANING
# ---------------------------------------------------------------

# Remove duplicate rows
df = df.drop_duplicates()

# Replace missing values using forward fill
df = df.fillna(method='ffill')

# If still any NaN, fill with column mean
df = df.fillna(df.mean(numeric_only=True))

# ---------------------------------------------------------------
# 4. CREATE FOG LABEL
# Fog is defined when visibility < 1 km
# 1 = Fog, 0 = No Fog
# ---------------------------------------------------------------
df['Fog'] = df['Visibility'].apply(lambda x: 1 if x < 1 else 0)

# ---------------------------------------------------------------
# 5. FEATURE ENGINEERING
# Temperature-DewPoint Spread (important for fog prediction)
# ---------------------------------------------------------------
df['Temp_Dew_Spread'] = df['Temperature'] - df['DewPoint']

# ---------------------------------------------------------------
# 6. SELECT FEATURES & TARGET
# ---------------------------------------------------------------
X = df[['Temperature', 'DewPoint', 'Visibility', 'Temp_Dew_Spread']]
y = df['Fog']

# ---------------------------------------------------------------
# 7. TRAIN-TEST SPLIT
# ---------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------------------------------------------------
# 8. FEATURE SCALING (Normalization)
# ---------------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------------------------------------------------------
# 9. TRAIN THE MODEL (Random Forest)
# ---------------------------------------------------------------
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train_scaled, y_train)

# ---------------------------------------------------------------
# 10. MAKE PREDICTIONS
# ---------------------------------------------------------------
y_pred = model.predict(X_test_scaled)

# ---------------------------------------------------------------
# 11. PERFORMANCE REPORT
# ---------------------------------------------------------------
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ---------------------------------------------------------------
# 12. EXAMPLE – PREDICT FOG FOR NEW VALUES
# ---------------------------------------------------------------
example = pd.DataFrame({
    "Temperature": [10],
    "DewPoint": [9],
    "Visibility": [0.4],
    "Temp_Dew_Spread": [10-9]
})

example_scaled = scaler.transform(example)
prediction = model.predict(example_scaled)

print("\nFog Prediction (1=Fog, 0=No Fog):", prediction[0])


Sample Data:
   Temperature   DewPoint  Visibility   Humidity  WindSpeed
0     9.981605  14.435660    1.932816  71.144907   5.234114
1    33.028572   8.763373    5.464819  68.750913   4.939576
2    24.279758   0.833467    8.742164  41.538524  18.125092
3    18.946339  18.482826    7.349026  60.474870   4.990924
4     1.240746  13.965591    8.084955  62.811737   5.438995


/tmp/ipython-input-2659150941.py:36: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='ffill')



Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        93
           1       1.00      1.00      1.00         7

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100


Fog Prediction (1=Fog, 0=No Fog): 1
